In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Flatten, Conv2D,  MaxPool2D, Dropout, BatchNormalization

import os
from tqdm import tqdm

from sklearn.metrics import accuracy_score

from PIL import Image
import gc

In [ ]:
train_path = r"Gender Classification Dataset\Training"
val_path = r"Gender Classification Dataset\Validation"

In [ ]:
def read_data(path, size=(50, 50)):
    classes = {}

    k = 0
    for p, fol, file in os.walk(path):
        for f in file:
            if f.endswith(".jpg"):
                k += 1

    images = np.zeros((k, *size, 3), dtype=np.float32)
    labels = np.zeros((k,), dtype=np.float32)

    ind = 0
    for i, label in enumerate(sorted(os.listdir(path))):
        classes[i] = label            
        for img_name in tqdm(os.listdir(os.path.join(path, label))):
            if not img_name.endswith(".jpg"):
                continue
            img_path = os.path.join(path, label, img_name)

            with Image.open(img_path) as img:
                img = np.asarray(img.convert("RGB").resize(size)) / 255.
            
            images[ind] = img
            labels[ind] = i
            ind += 1

            if ind % 1000 == 0:
                gc.collect()

    return classes, images, labels

In [ ]:
classes, x_train, y_train = read_data(train_path, size=(50, 50))

In [ ]:
classes

In [ ]:
classes, x_test, y_test = read_data(val_path, size=(50, 50))

In [ ]:
classes

In [ ]:
plt.figure(figsize=(12, 15))
for i in range(100):
    plt.subplot(10, 10, i + 1)
    ind = np.random.randint(0, len(x_train) - 1)
    plt.imshow(x_train[ind])
    plt.xticks([])
    plt.yticks([])
    plt.title(classes[y_train[ind]])

In [ ]:
y_train_cat = tf.keras.utils.to_categorical(y_train)
y_test_cat = tf.keras.utils.to_categorical(y_test)

In [ ]:
model = tf.keras.Sequential([
    Input(shape=x_train.shape[1:]),

    Conv2D(64, 3, padding="same", activation="relu"),
    Conv2D(64, 3, padding="same", activation="relu"),
    MaxPool2D(2),

    Conv2D(128, 3, padding="same", activation="relu"),
    Conv2D(128, 3, padding="same", activation="relu"),
    MaxPool2D(2),

    Conv2D(128, 3, padding="same", activation="relu"),
    Conv2D(128, 3, padding="same", activation="relu"),
    MaxPool2D(2),

    Flatten(),
    Dense(64, activation="relu"),
    Dense(2, activation="softmax")
])
model.summary()

In [ ]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
model.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=3, batch_size=32)